# Titanic Dual-Task Project — Classification & Regression

**Name:** Nixan Thapa  
**Roll Number:** ACE081BCT049  

**Dataset:** `titanic` (seaborn built-in)  
**Tasks & Models:**
- **Classification:** Predict `survived` — Logistic Regression vs Random Forest Classifier  
- **Regression:** Predict `fare` — Linear Regression vs Random Forest Regressor  

**Structure:**
1. Problem definition  
2. Data overview & EDA  
3. Feature engineering  
4. Preprocessing  
5. Train-test splits  
6. Classification — training, CV, evaluation, ROC curve  
7. Regression — training, CV, evaluation, residual plot  
8. Interpretation & conclusion  


## 1) Problem Definition

We solve two supervised-learning problems on the same Titanic dataset.

**A. Classification — Logistic Regression vs Random Forest Classifier**  
- **Objective:** Predict whether a passenger survived (`survived` = 0/1).  
- **Baseline:** Logistic Regression — interpretable, fast, strong linear baseline.  
- **Challenger:** Random Forest — captures non-linear interactions and feature interactions.

**B. Regression — Linear Regression vs Random Forest Regressor**  
- **Objective:** Predict the ticket fare a passenger paid (`fare`, continuous).  
- **Baseline:** Linear Regression — interpretable coefficients, good first-order model.  
- **Challenger:** Random Forest Regressor — handles skewness and non-linearities.  
- We test two target encodings: raw-capped `fare_capped` and `log(fare+1)`.

Both tasks share a feature pipeline. The `fare` column is **never** included as a feature  
(it is the regression target and would cause data leakage if used for classification too).  


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import clone

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay,
    r2_score, mean_squared_error
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = sns.load_dataset('titanic')
print('Dataset shape:', df.shape)
df.head()


## 2) Data Overview & EDA

In [ ]:
# ── 2a  Data types & missing values ─────────────────────────────────────────
print('Dtypes:\n', df.dtypes, '\n')
print('Missing values (selected columns):')
cols_of_interest = ['survived','pclass','sex','age','sibsp','parch','fare','embarked']
print(df[cols_of_interest].isnull().sum())
print('\nClass balance (survived):')
print(df['survived'].value_counts(normalize=True).rename({0:'died',1:'survived'}))
print('\nFare skewness:', round(df['fare'].skew(), 3))


In [ ]:
# ── 2b  Survival patterns ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Survival rate by sex
sex_surv = df.groupby('sex')['survived'].mean().reset_index()
sns.barplot(data=sex_surv, x='sex', y='survived', ax=axes[0], palette='Set2')
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', fontsize=11)

# Survival rate by pclass
cls_surv = df.groupby('pclass')['survived'].mean().reset_index()
sns.barplot(data=cls_surv, x='pclass', y='survived', ax=axes[1], palette='Set1')
axes[1].set_title('Survival Rate by Passenger Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', fontsize=11)

# Survival rate by embarked
emb_surv = df.groupby('embarked')['survived'].mean().reset_index()
sns.barplot(data=emb_surv, x='embarked', y='survived', ax=axes[2], palette='Set3')
axes[2].set_title('Survival Rate by Embarkation Port')
axes[2].set_ylabel('Survival Rate')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.suptitle('Key Categorical Features vs Survival', y=1.02, fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# ── 2c  Age & fare distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Age distribution split by survival
survived   = df[df['survived'] == 1]['age'].dropna()
not_survived = df[df['survived'] == 0]['age'].dropna()
axes[0].hist(not_survived, bins=30, alpha=0.6, color='tomato',   label='Did not survive')
axes[0].hist(survived,     bins=30, alpha=0.6, color='steelblue', label='Survived')
axes[0].set_title('Age Distribution by Survival')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Fare histogram + log-scale comparison
axes[1].hist(df['fare'].dropna(), bins=50, color='mediumpurple', alpha=0.8)
axes[1].set_title('Fare Distribution (highly skewed)')
axes[1].set_xlabel('Fare')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Correlation heatmap (numeric features only)
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
plt.figure(figsize=(7, 5))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()


## 3) Feature Engineering

Three new features are derived from the existing columns:

| Feature | Formula | Rationale |
|---|---|---|
| `family_size` | `sibsp + parch + 1` | Larger families may have lower survival (harder to evacuate together) |
| `is_alone` | 1 if `family_size == 1` else 0 | Travelling alone is a distinct social situation |
| `age_band` | 4 equal-width bins over age | Captures non-linear age effects (children, adults, elderly) |

`fare` is **excluded** from all feature sets — it is the regression target.


In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone']    = (df['family_size'] == 1).astype(int)
df['age_band']    = pd.cut(df['age'], bins=[0, 12, 18, 60, 100],
                            labels=['child','teen','adult','senior'])

print('New features added:')
print(df[['family_size','is_alone','age_band']].head(8))
print('\nFamily size distribution:')
print(df['family_size'].value_counts().sort_index())
print('\nIs alone:', df['is_alone'].value_counts().to_dict())

# Visualise family size vs survival
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fs_surv = df.groupby('family_size')['survived'].mean().reset_index()
sns.barplot(data=fs_surv, x='family_size', y='survived', ax=axes[0], palette='viridis')
axes[0].set_title('Survival Rate by Family Size')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)

age_surv = df.groupby('age_band', observed=True)['survived'].mean().reset_index()
sns.barplot(data=age_surv, x='age_band', y='survived', ax=axes[1], palette='magma')
axes[1].set_title('Survival Rate by Age Band')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 4) Preprocessing

Steps:
- **Numeric features** (`age`, `sibsp`, `parch`, `family_size`, `is_alone`): median imputation → StandardScaler.  
- **Categorical features** (`pclass`, `sex`, `embarked`, `age_band`): most-frequent imputation → OneHotEncoder.  
- **Important:** a *fresh clone* of the preprocessor is passed to each pipeline so that fitting one  
  does not corrupt the state of the other (a subtle but real bug when sharing the same object).


In [ ]:
# ── Preprocessing definition ─────────────────────────────────────────────────
features = ['pclass','sex','age','sibsp','parch','embarked',
            'family_size','is_alone','age_band']

clf_target = 'survived'
reg_target = 'fare'

numeric_features     = ['age','sibsp','parch','family_size','is_alone']
categorical_features = ['pclass','sex','embarked','age_band']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

def make_preprocessor():
    """Return a fresh ColumnTransformer each call to avoid shared-state issues."""
    return ColumnTransformer(transformers=[
        ('num', clone(numeric_transformer),     numeric_features),
        ('cat', clone(categorical_transformer), categorical_features)
    ])

print('Preprocessor factory defined.')
print('Numeric features :', numeric_features)
print('Categorical features:', categorical_features)


In [ ]:
# ── Prepare modelling data ───────────────────────────────────────────────────
def cap_outliers_iqr(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr    = q3 - q1
    return series.clip(lower=q1 - 1.5*iqr, upper=q3 + 1.5*iqr)

data = df[features + [clf_target, reg_target]].copy()

# Regression targets: capped fare and log(fare+1)
data['fare_capped'] = cap_outliers_iqr(data[reg_target].fillna(data[reg_target].median()))
data['fare_log']    = np.log1p(data[reg_target].fillna(data[reg_target].median()))

print('fare_capped stats:'); print(data['fare_capped'].describe().round(2))
print('\nfare_log stats:');   print(data['fare_log'].describe().round(2))


## 5) Train-Test Splits

In [ ]:
# ── Train-test splits ────────────────────────────────────────────────────────
X_clf = data[features].copy()
y_clf = data[clf_target].copy()

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print('Classification split:', X_clf_train.shape, X_clf_test.shape)

reg_data   = data[features + ['fare_capped','fare_log']].dropna(subset=['fare_capped'])
X_reg      = reg_data[features].copy()
y_reg_cap  = reg_data['fare_capped'].copy()
y_reg_log  = reg_data['fare_log'].copy()

X_reg_train, X_reg_test, y_cap_train, y_cap_test, y_log_train, y_log_test = train_test_split(
    X_reg, y_reg_cap, y_reg_log, test_size=0.2, random_state=42
)
print('Regression split:', X_reg_train.shape, X_reg_test.shape)


## 6) Classification — Logistic Regression vs Random Forest

Both models use the same preprocessing pipeline (independent instances via `make_preprocessor()`).  
Evaluation: 5-fold stratified CV accuracy, then hold-out Accuracy / Precision / Recall / AUC.


In [ ]:
# ── Build & cross-validate classifiers ──────────────────────────────────────
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf_lr = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('clf', LogisticRegression(solver='liblinear', random_state=42))
])

clf_rf = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=8,
                                    random_state=42, n_jobs=-1))
])

cv_lr_acc = cross_val_score(clf_lr, X_clf, y_clf, cv=cv_strat, scoring='accuracy')
cv_rf_acc = cross_val_score(clf_rf, X_clf, y_clf, cv=cv_strat, scoring='accuracy')

print('=== 5-Fold Stratified CV (Accuracy) ===')
print(f'Logistic Regression : {cv_lr_acc.mean():.4f}  ± {cv_lr_acc.std():.4f}')
print(f'Random Forest       : {cv_rf_acc.mean():.4f}  ± {cv_rf_acc.std():.4f}')


In [ ]:
# ── Fit on train set, evaluate on hold-out ───────────────────────────────────
clf_lr.fit(X_clf_train, y_clf_train)
clf_rf.fit(X_clf_train, y_clf_train)

y_lr_pred  = clf_lr.predict(X_clf_test)
y_rf_pred  = clf_rf.predict(X_clf_test)
y_lr_proba = clf_lr.predict_proba(X_clf_test)[:, 1]
y_rf_proba = clf_rf.predict_proba(X_clf_test)[:, 1]

def clf_metrics(y_true, y_pred, y_proba):
    return {
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'AUC-ROC'  : round(roc_auc_score(y_true, y_proba), 4),
    }

results_clf = pd.DataFrame([
    {'Model': 'Logistic Regression', **clf_metrics(y_clf_test, y_lr_pred, y_lr_proba)},
    {'Model': 'Random Forest',       **clf_metrics(y_clf_test, y_rf_pred, y_rf_proba)},
]).set_index('Model')

print('=== Hold-out Classification Results ===')
print(results_clf)
print()
print('Logistic Regression Report:')
print(classification_report(y_clf_test, y_lr_pred, zero_division=0))


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, model, preds, title in zip(
        axes,
        [clf_lr, clf_rf],
        [y_lr_pred, y_rf_pred],
        ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_clf_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Died','Survived'], yticklabels=['Died','Survived'])
    ax.set_title(f'{title}\nConfusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_clf_test, y_lr_proba,
                                  name=f'Logistic Regression (AUC={roc_auc_score(y_clf_test,y_lr_proba):.3f})',
                                  ax=ax, color='steelblue')
RocCurveDisplay.from_predictions(y_clf_test, y_rf_proba,
                                  name=f'Random Forest       (AUC={roc_auc_score(y_clf_test,y_rf_proba):.3f})',
                                  ax=ax, color='darkorange')
ax.plot([0,1],[0,1],'k--', label='Random Classifier (AUC=0.500)')
ax.set_title('ROC Curves — Classification Task')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Logistic Regression coefficients ─────────────────────────────────────────
ohe_lr    = clf_lr.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
feat_names_cat = list(ohe_lr.get_feature_names_out(categorical_features))
feat_names_all = numeric_features + feat_names_cat

coef_lr_df = pd.DataFrame({
    'feature'    : feat_names_all,
    'coefficient': clf_lr.named_steps['clf'].coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

plt.figure(figsize=(8, 5))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_lr_df['coefficient']]
plt.barh(coef_lr_df['feature'], coef_lr_df['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients\n(positive = higher survival odds)')
plt.xlabel('Coefficient')
plt.tight_layout()
plt.show()

# ── Random Forest feature importance ─────────────────────────────────────────
ohe_rf   = clf_rf.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
feat_rf  = numeric_features + list(ohe_rf.get_feature_names_out(categorical_features))
imp_df   = pd.DataFrame({
    'feature'   : feat_rf,
    'importance': clf_rf.named_steps['clf'].feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
plt.title('Random Forest Feature Importances (Classification)')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()


## 7) Regression — Linear Regression vs Random Forest Regressor

Targets tested:
- **`fare_capped`** — IQR-capped raw fare.  
- **`log(fare+1)`** — log-transform; predictions are inverted with `expm1` for interpretable RMSE.

Evaluation: 5-fold CV (neg-RMSE), then hold-out R² / MSE / RMSE.


In [ ]:
# ── Build & cross-validate regressors ────────────────────────────────────────
cv_kf = KFold(n_splits=5, shuffle=True, random_state=42)

reg_lr_cap = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('reg', LinearRegression())
])
reg_rf_cap = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('reg', RandomForestRegressor(n_estimators=200, max_depth=8,
                                   random_state=42, n_jobs=-1))
])
reg_lr_log = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('reg', LinearRegression())
])
reg_rf_log = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('reg', RandomForestRegressor(n_estimators=200, max_depth=8,
                                   random_state=42, n_jobs=-1))
])

def cv_rmse(pipeline, X, y):
    scores = cross_val_score(pipeline, X, y, cv=cv_kf, scoring='neg_root_mean_squared_error')
    return -scores.mean(), scores.std()

print('=== 5-Fold CV RMSE (fare_capped target) ===')
m, s = cv_rmse(reg_lr_cap, X_reg, y_reg_cap)
print(f'  Linear Regression : {m:.4f} ± {s:.4f}')
m, s = cv_rmse(reg_rf_cap, X_reg, y_reg_cap)
print(f'  Random Forest     : {m:.4f} ± {s:.4f}')

print()
print('=== 5-Fold CV RMSE (log-fare target) ===')
m, s = cv_rmse(reg_lr_log, X_reg, y_reg_log)
print(f'  Linear Regression : {m:.4f} ± {s:.4f}')
m, s = cv_rmse(reg_rf_log, X_reg, y_reg_log)
print(f'  Random Forest     : {m:.4f} ± {s:.4f}')


In [ ]:
# ── Fit all regression models on train set ───────────────────────────────────
for pipe, Xtr, ytr in [
    (reg_lr_cap, X_reg_train, y_cap_train),
    (reg_rf_cap, X_reg_train, y_cap_train),
    (reg_lr_log, X_reg_train, y_log_train),
    (reg_rf_log, X_reg_train, y_log_train),
]:
    pipe.fit(Xtr, ytr)

# Hold-out evaluation helper
def reg_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {
        'R²'  : round(r2_score(y_true, y_pred), 4),
        'MSE' : round(mean_squared_error(y_true, y_pred), 4),
        'RMSE': round(rmse, 4),
    }

# For log models: invert to original scale before comparing RMSE
y_lr_cap_pred  = reg_lr_cap.predict(X_reg_test)
y_rf_cap_pred  = reg_rf_cap.predict(X_reg_test)
y_lr_log_pred  = np.expm1(reg_lr_log.predict(X_reg_test))
y_rf_log_pred  = np.expm1(reg_rf_log.predict(X_reg_test))
y_test_orig    = np.expm1(y_log_test)   # original-scale actuals

results_reg = pd.DataFrame([
    {'Model': 'LR  — fare_capped',   **reg_metrics(y_cap_test, y_lr_cap_pred)},
    {'Model': 'RF  — fare_capped',   **reg_metrics(y_cap_test, y_rf_cap_pred)},
    {'Model': 'LR  — log(fare+1)*',  **reg_metrics(y_test_orig, y_lr_log_pred)},
    {'Model': 'RF  — log(fare+1)*',  **reg_metrics(y_test_orig, y_rf_log_pred)},
]).set_index('Model')

print('=== Hold-out Regression Results ===')
print('* log-model metrics are on the original fare scale (predictions inverted)')
print()
print(results_reg)


In [ ]:
# ── Residual plots ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title, y_true in [
    (axes[0], y_lr_cap_pred, 'Linear Regression — fare_capped', y_cap_test),
    (axes[1], y_rf_cap_pred, 'Random Forest     — fare_capped', y_cap_test),
]:
    residuals = y_true - y_pred
    ax.scatter(y_pred, residuals, alpha=0.4, edgecolors='none', color='steelblue', s=20)
    ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Predicted Fare')
    ax.set_ylabel('Residual (Actual − Predicted)')

plt.suptitle('Residual Plots — Regression Task', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Regression feature importances (Random Forest) ───────────────────────────
ohe_reg  = reg_rf_cap.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
feat_reg = numeric_features + list(ohe_reg.get_feature_names_out(categorical_features))

imp_reg_df = pd.DataFrame({
    'feature'   : feat_reg,
    'importance': reg_rf_cap.named_steps['reg'].feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=imp_reg_df, x='importance', y='feature', palette='mako')
plt.title('Random Forest Feature Importances (Regression)')
plt.xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()


## 8) Interpretation & Conclusion

### Classification

| Model | CV Accuracy | Hold-out Accuracy | AUC-ROC |
|---|---|---|---|
| Logistic Regression | see above | see above | see above |
| Random Forest | see above | see above | see above |

**Key findings from coefficients / importance:**
- `sex_female` has the strongest positive effect on survival (log-odds).
- Higher `pclass` (3rd class) is strongly negative.
- `family_size` and `is_alone` both contribute meaningful signal beyond the raw `sibsp`/`parch`.
- Random Forest captures non-linear interactions missed by Logistic Regression (especially class × sex interactions).

### Regression

| Model | Target | CV RMSE | Hold-out R² | Hold-out RMSE |
|---|---|---|---|---|
| Linear Regression | fare_capped | see above | see above | see above |
| Random Forest | fare_capped | see above | see above | see above |
| Linear Regression | log(fare+1) | see above | see above (orig scale) | see above |
| Random Forest | log(fare+1) | see above | see above (orig scale) | see above |

**Key findings:**
- `pclass` is the dominant predictor of fare — first-class tickets were significantly more expensive.
- The log-transform of fare reduces skewness and tends to improve linear regression's R².
- Random Forest achieves substantially better R² than Linear Regression by capturing non-linearities in class and embarkation port.

### Limitations & next steps

1. **Data leakage check** — confirmed: `fare` is excluded from all feature sets used for classification.
2. **Hyperparameter tuning** — apply `GridSearchCV` with stratified folds; try `C` for LR and `max_depth`/`n_estimators` for RF.
3. **More feature engineering** — `deck` has many missing values but encodes cabin location; impute with "Unknown" and one-hot encode.
4. **Class imbalance** — ~62% class-0; consider `class_weight='balanced'` or SMOTE if recall on survivors matters more.
5. **Ensemble / stacking** — blend LR and RF predictions for potentially higher AUC.
6. **Regression target** — test `QuantileTransformer` in addition to log; also try Ridge/Lasso to regularise Linear Regression.
